# MLP: representações aprendidas, forward e retropropagação

Este notebook acompanha a seção **Redes Neurais e Representações Aprendidas** da Nota 07.

Implementaremos, apenas com NumPy:

- um neurônio e uma camada oculta com ReLU;
- uma camada de saída com logits, softmax e entropia cruzada;
- o cálculo de gradientes pela regra da cadeia;
- a atualização de pesos por gradiente descendente;
- uma solução de XOR que evidencia por que a não linearidade é necessária.

Os exemplos são didáticos e sintéticos. O objetivo não é obter um framework de redes neurais, mas tornar visíveis as operações que bibliotecas como PyTorch e Keras automatizam.

## 1. Ferramentas básicas

A rede receberá uma matriz `X`, em que cada linha é um exemplo e cada coluna é uma característica. Para uma camada oculta, calculamos:

$$
U = X(W^{(1)})^\top + b^{(1)}, \qquad H = \operatorname{ReLU}(U).
$$

Em seguida, a camada de saída produz logits e probabilidades:

$$
Z = H(W^{(2)})^\top + b^{(2)}, \qquad P = \operatorname{softmax}(Z).
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
np.set_printoptions(precision=3, suppress=True)


def relu(u):
    return np.maximum(0.0, u)


def relu_grad(u):
    # Convenção usual: derivada zero no ponto u = 0.
    return (u > 0.0).astype(float)


def softmax(logits):
    # Subtrair o máximo torna o cálculo numericamente estável.
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(shifted)
    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)


def cross_entropy(probabilities, y):
    """Média de -log p(classe observada) para rótulos inteiros."""
    n = len(y)
    return -np.mean(np.log(probabilities[np.arange(n), y] + 1e-12))


def one_hot(y, n_classes):
    encoded = np.zeros((len(y), n_classes))
    encoded[np.arange(len(y)), y] = 1.0
    return encoded

## 2. Exemplo da nota: da entrada à perda

Usaremos exatamente a rede pequena apresentada na nota. Há duas características de entrada, dois neurônios ocultos com ReLU e duas classes de saída.

$$
x = \begin{bmatrix}1\\2\end{bmatrix}, \quad
W^{(1)}=\begin{bmatrix}1&-1\\1&1\end{bmatrix}, \quad
b^{(1)}=\begin{bmatrix}0\\0\end{bmatrix}.
$$

A primeira linha de `W1` é o vetor de pesos do primeiro neurônio oculto. Neste notebook, mantemos exemplos nas linhas; por isso, multiplicamos por `W1.T`.

In [ ]:
# Um único exemplo, armazenado como uma matriz de uma linha.
X_example = np.array([[1.0, 2.0]])

W1 = np.array([[1.0, -1.0],
               [1.0,  1.0]])
b1 = np.array([0.0, 0.0])

W2 = np.array([[1.0, 1.0],
               [0.0, 0.0]])
b2 = np.array([0.0, 0.0])

U = X_example @ W1.T + b1
H = relu(U)
Z = H @ W2.T + b2
P = softmax(Z)

print("U = X W1^T + b1 =", U)
print("H = ReLU(U)     =", H)
print("Z = H W2^T + b2 =", Z)
print("P = softmax(Z)  =", P)

A ReLU transforma $(-1, 3)$ em $(0, 3)$. A camada de saída produz logits $(3, 0)$, que a softmax converte em probabilidades. A decisão de classe é a de maior probabilidade, mas a perda também depende da confiança atribuída à classe observada.

In [ ]:
for target in (0, 1):
    loss = cross_entropy(P, np.array([target]))
    print(f"Alvo = classe {target}: perda = {loss:.3f}")

print("\nClasse prevista:", np.argmax(P, axis=1)[0])
print("Probabilidade da classe 0:", P[0, 0])

Quando o alvo é a classe 0, a perda é baixa porque a rede atribuiu alta probabilidade a essa classe. Se o alvo observado for a classe 1, a mesma previsão gera perda alta. Isso é o que a entropia cruzada penaliza.

## 3. Retropropagação: da perda aos gradientes

Para softmax combinada com entropia cruzada, a derivada em relação aos logits de cada exemplo tem uma forma simples:

$$
\frac{\partial \ell}{\partial z} = p - y_{\mathrm{one\ hot}}.
$$

A partir dela, a regra da cadeia propaga o gradiente para a camada de saída, para a representação oculta e para a primeira camada. A célula abaixo calcula todos esses termos para o exemplo anterior.

In [ ]:
target = np.array([0])
target_one_hot = one_hot(target, n_classes=2)

# Gradiente da perda em relação aos logits.
dZ = P - target_one_hot

# Gradientes dos parâmetros da segunda camada.
dW2 = dZ.T @ H
db2 = dZ.sum(axis=0)

# Propagação do gradiente para a camada oculta.
dH = dZ @ W2
dU = dH * relu_grad(U)

# Gradientes dos parâmetros da primeira camada.
dW1 = dU.T @ X_example
db1 = dU.sum(axis=0)

print("dL/dZ =\n", dZ)
print("\ndL/dW2 =\n", dW2)
print("dL/db2 =", db2)
print("\ndL/dW1 =\n", dW1)
print("dL/db1 =", db1)

print("\nDerivada destacada dL/dW2[0, 1]:", dW2[0, 1])

A derivada destacada corresponde ao peso que liga $h_2=3$ ao primeiro logit. Seu valor é aproximadamente $-0{,}142$, como na nota. O sinal negativo informa a direção local: aumentar esse peso reduz a perda deste exemplo, mantendo os demais parâmetros fixos.

In [ ]:
def forward(X, W1, b1, W2, b2):
    U = X @ W1.T + b1
    H = relu(U)
    Z = H @ W2.T + b2
    P = softmax(Z)
    return U, H, Z, P


learning_rate = 0.1
loss_before = cross_entropy(P, target)

# Atualização simultânea: todos os gradientes foram calculados com os parâmetros atuais.
W1_new = W1 - learning_rate * dW1
b1_new = b1 - learning_rate * db1
W2_new = W2 - learning_rate * dW2
b2_new = b2 - learning_rate * db2

_, _, _, P_after = forward(X_example, W1_new, b1_new, W2_new, b2_new)
loss_after = cross_entropy(P_after, target)

print(f"Perda antes da atualização: {loss_before:.4f}")
print(f"Perda depois da atualização: {loss_after:.4f}")
print("Novo W2[0, 1]:", W2_new[0, 1])

Uma atualização pode reduzir a perda deste exemplo sem garantir redução da perda de todo o conjunto ou da validação. Por isso, treinamento, validação e teste continuam tendo papéis distintos.

## 4. XOR: uma camada oculta transforma a representação

XOR é um caso clássico: os pontos positivos $(0,1)$ e $(1,0)$ ocupam cantos alternados em relação aos negativos $(0,0)$ e $(1,1)$. Nenhuma reta os separa corretamente no espaço original.

A nota apresenta uma camada oculta com:

$$
h_1=\operatorname{ReLU}(x_1+x_2-1), \qquad
h_2=\operatorname{ReLU}(1-x_1-x_2).
$$

Ela transforma os dois exemplos positivos no mesmo ponto $(0,0)$ e permite uma separação linear no espaço oculto.

In [ ]:
X_xor = np.array([[0.0, 0.0],
                  [0.0, 1.0],
                  [1.0, 0.0],
                  [1.0, 1.0]])
y_xor = np.array([0, 1, 1, 0])  # 1 = classe positiva

W1_xor = np.array([[ 1.0,  1.0],
                   [-1.0, -1.0]])
b1_xor = np.array([-1.0, 1.0])

H_xor = relu(X_xor @ W1_xor.T + b1_xor)
score_xor = 0.5 - H_xor[:, 0] - H_xor[:, 1]
prediction_xor = (score_xor > 0.0).astype(int)

print("Entrada | classe | representação oculta | escore | previsão")
for x, label, h, score, pred in zip(X_xor, y_xor, H_xor, score_xor, prediction_xor):
    print(f"{x.astype(int)}      {label}          {h}       {score:5.2f}      {pred}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)

colors = np.where(y_xor == 1, "tab:green", "tab:red")
markers = np.where(y_xor == 1, "o", "s")

for x, label, color, marker in zip(X_xor, y_xor, colors, markers):
    axes[0].scatter(*x, s=130, color=color, marker=marker, edgecolor="black", zorder=3)
    axes[0].annotate("+" if label else "−", x, xytext=(6, 6), textcoords="offset points")

axes[0].set(title="Espaço de entrada", xlabel="$x_1$", ylabel="$x_2$",
            xlim=(-0.2, 1.2), ylim=(-0.2, 1.2), aspect="equal")

for h, label, color, marker in zip(H_xor, y_xor, colors, markers):
    axes[1].scatter(*h, s=130, color=color, marker=marker, edgecolor="black", zorder=3)
    axes[1].annotate("+" if label else "−", h, xytext=(6, 6), textcoords="offset points")

grid = np.linspace(-0.1, 1.15, 100)
axes[1].plot(grid, 0.5 - grid, "--", color="tab:blue", label="$h_1+h_2=0,5$")
axes[1].set(title="Representação oculta", xlabel="$h_1$", ylabel="$h_2$",
            xlim=(-0.2, 1.2), ylim=(-0.2, 1.2), aspect="equal")
axes[1].legend()
plt.show()

A fronteira $h_1+h_2=0{,}5$ é linear no espaço oculto. No espaço original, a decisão correspondente não é uma única reta. A camada oculta mudou a representação para que a camada de saída pudesse resolver a tarefa com uma operação simples.

## 5. Treinando um MLP para XOR, do zero

Agora os mesmos tipos de parâmetros serão aprendidos, e não definidos manualmente. A implementação usa uma camada oculta ReLU com quatro neurônios e uma saída de duas classes.

A sequência em cada época é:

1. **forward:** calcular $U$, $H$, $Z$, $P$ e a perda;
2. **backward:** calcular gradientes pela regra da cadeia;
3. **update:** atualizar todos os pesos e vieses simultaneamente.

In [ ]:
def mlp_forward(X, params):
    U1 = X @ params["W1"].T + params["b1"]
    H1 = relu(U1)
    Z2 = H1 @ params["W2"].T + params["b2"]
    P2 = softmax(Z2)
    cache = {"X": X, "U1": U1, "H1": H1, "P2": P2}
    return P2, cache


def mlp_gradients(cache, y, params):
    X, U1, H1, P2 = cache["X"], cache["U1"], cache["H1"], cache["P2"]
    n = len(y)
    dZ2 = (P2 - one_hot(y, P2.shape[1])) / n
    dW2 = dZ2.T @ H1
    db2 = dZ2.sum(axis=0)

    dH1 = dZ2 @ params["W2"]
    dU1 = dH1 * relu_grad(U1)
    dW1 = dU1.T @ X
    db1 = dU1.sum(axis=0)
    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}


def initialize_mlp(n_inputs, n_hidden, n_classes, seed=0):
    rng = np.random.default_rng(seed)
    return {
        "W1": rng.normal(0, 0.7, size=(n_hidden, n_inputs)),
        "b1": np.zeros(n_hidden),
        "W2": rng.normal(0, 0.7, size=(n_classes, n_hidden)),
        "b2": np.zeros(n_classes),
    }


def train_mlp(X, y, n_hidden=4, learning_rate=0.1, epochs=4000, seed=7):
    params = initialize_mlp(X.shape[1], n_hidden, len(np.unique(y)), seed)
    history = []

    for epoch in range(epochs):
        probabilities, cache = mlp_forward(X, params)
        history.append(cross_entropy(probabilities, y))
        gradients = mlp_gradients(cache, y, params)

        for name in params:
            params[name] -= learning_rate * gradients[name]

    return params, np.array(history)

In [ ]:
params_xor, loss_history = train_mlp(
    X_xor, y_xor, n_hidden=4, learning_rate=0.1, epochs=4000, seed=7
)

probabilities_xor, cache_xor = mlp_forward(X_xor, params_xor)
predictions_xor = np.argmax(probabilities_xor, axis=1)

print("Probabilidades finais [classe 0, classe 1]:")
print(probabilities_xor)
print("\nPrevisões:", predictions_xor)
print("Rótulos:  ", y_xor)
print("Acurácia:", np.mean(predictions_xor == y_xor))

In [ ]:
plt.figure(figsize=(7, 3.5))
plt.plot(loss_history, color="tab:blue")
plt.xlabel("Época")
plt.ylabel("Entropia cruzada")
plt.title("Perda de treinamento do MLP no XOR")
plt.yscale("log")
plt.show()

In [ ]:
def plot_decision_regions(X, y, params, title):
    x1 = np.linspace(-0.25, 1.25, 250)
    x2 = np.linspace(-0.25, 1.25, 250)
    xx1, xx2 = np.meshgrid(x1, x2)
    grid = np.column_stack([xx1.ravel(), xx2.ravel()])

    probabilities, _ = mlp_forward(grid, params)
    predicted_class = np.argmax(probabilities, axis=1).reshape(xx1.shape)

    plt.figure(figsize=(5.5, 4.5))
    plt.contourf(xx1, xx2, predicted_class, levels=[-0.5, 0.5, 1.5],
                 colors=["#f5c2c2", "#c8e6c9"], alpha=0.8)
    plt.contour(xx1, xx2, probabilities[:, 1].reshape(xx1.shape),
                levels=[0.5], colors="tab:blue", linewidths=2)

    for x, label in zip(X, y):
        plt.scatter(*x, color="tab:green" if label else "tab:red",
                    edgecolor="black", s=120, zorder=3)
        plt.annotate("+" if label else "−", x, xytext=(6, 6),
                     textcoords="offset points")

    plt.xlim(-0.25, 1.25)
    plt.ylim(-0.25, 1.25)
    plt.xlabel("$x_1$")
    plt.ylabel("$x_2$")
    plt.title(title)
    plt.show()


plot_decision_regions(X_xor, y_xor, params_xor,
                      "Fronteira aprendida pelo MLP para XOR")

## 6. E se retirarmos a ReLU?

Sem ativação entre as camadas:

$$
Z = (X(W^{(1)})^\top+b^{(1)})(W^{(2)})^\top+b^{(2)}
  = X\widetilde W^\top+\widetilde b.
$$

Duas transformações afins consecutivas ainda são uma única transformação afim. A profundidade, por si só, não torna os logits não lineares em relação à entrada. A softmax converte logits em probabilidades, mas não cria a transformação oculta não linear que falta para separar XOR.

In [ ]:
# A composição de duas camadas lineares pode ser reduzida a uma só.
rng = np.random.default_rng(1734)
W1_linear = rng.normal(size=(3, 2))
b1_linear = rng.normal(size=3)
W2_linear = rng.normal(size=(2, 3))
b2_linear = rng.normal(size=2)

W_equivalent = W2_linear @ W1_linear
b_equivalent = W2_linear @ b1_linear + b2_linear

X_check = rng.normal(size=(5, 2))
two_linear_layers = (X_check @ W1_linear.T + b1_linear) @ W2_linear.T + b2_linear
one_linear_layer = X_check @ W_equivalent.T + b_equivalent

print("As duas formas produzem o mesmo resultado?",
      np.allclose(two_linear_layers, one_linear_layer))
print("Maior diferença numérica:", np.max(np.abs(two_linear_layers - one_linear_layer)))

## 7. Representações aprendidas e generalização

No exemplo resolvido da nota, a representação oculta de $(1,2)$ era $(0,3)$ para uma configuração dos parâmetros. Durante o treinamento completo, os pesos da primeira camada também são atualizados. Portanto, a representação de uma mesma entrada pode mudar para ajudar a reduzir a perda da tarefa.

Mais capacidade não elimina a necessidade de validação. Em uma aplicação real, deve-se separar dados de treinamento, validação e teste; acompanhar a perda de validação; e guardar os parâmetros da melhor época de validação (*early stopping*).

In [ ]:
# A mesma entrada produz outra representação após a atualização do exemplo.
_, H_before, _, _ = forward(X_example, W1, b1, W2, b2)
_, H_after, _, _ = forward(X_example, W1_new, b1_new, W2_new, b2_new)

print("Representação antes da atualização:", H_before)
print("Representação depois da atualização:", H_after)
print("\nNo treinamento de um MLP, o gradiente alcança W1 e b1; por isso,")
print("a camada oculta pode aprender representações úteis para a saída.")

## Exercícios e discussões

1. Na arquitetura $4 \rightarrow 5 \rightarrow 2$ da nota, quantos parâmetros há? Se a saída passar a ter três classes, quais dimensões e quantidades mudam?
2. No XOR, por que mapear os dois exemplos positivos para o mesmo ponto não impede a classificação?
3. Modifique `n_hidden`, `learning_rate` ou `seed` no treinamento do XOR. A perda sempre diminui da mesma forma? O que o experimento mostra sobre otimização?
4. Remova a ReLU em `mlp_forward` e ajuste a retropropagação de modo coerente. O que acontece com a capacidade de separar XOR?
5. Um MLP apresenta perda de treino decrescente e perda de validação crescente. Que parâmetros você guardaria? Por que o teste não deve escolher a época final?